# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MTahaR-dev/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane lock: I confirm Lane 2 — Refresh / Content Opportunity Scoring.** My decision remains which pages a content editor should review first for a possible refresh. A recommendation requests investigation; it does not establish that an edit will recover traffic.

This notebook audits two signals, implements one transparent rule, regenerates `work/outputs/baseline_action_score.csv`, and reviews its first ten recommendations. It writes an aggregate metrics receipt to `work/outputs/w04_baseline_metrics.json`. The CSV and raw data stay out of Git; the notebook and metrics JSON are deliverables.

I used AI assistance with the repo's `building-baselines` and `flyrank/flyrank-data` skills. The top-ten review checks measured evidence and plausible failure modes, not article content: text, URLs, and intervention history are unavailable.

**Rerun:** Run all cells with Python 3. Reuse the local March parquet file or securely configure a Hugging Face READ token as described in Week 3. Only March is read; June remains untouched. Windows and the historical decision date match the Week 3 contract.

## 1. Two signal checks and my rule's reasoning

**Time discipline:** Source grain is report date × client × content. The scoring grain is one client/content item at **March 18, 2026**. Inputs use **March 1–14**; March 15–17 is the reporting buffer. Within the input window, week 1 is March 1–7 and week 2 is March 8–14. Future labels use March 18–31 and appear only in separate audit/evaluation data. Historical ingestion times are unknown, so the buffer remains an assumption.

The action population requires 14 usable pre-decision GSC days and at least 100 input-window impressions, without consulting future coverage. Labels additionally require 14 usable later days, matching Week 3's complete-case evaluation. The production-style queue does not impose that later requirement.

**Audit design:** I use the same seed-42 whole-client split as Week 3. Bucket tables use development clients only, before the rule is frozen. Test clients are used only for the final baseline evaluation. No threshold search or fitted weight is used.

**Signal A — exposure/volume:** The assignment links volume to FlyRank's quick-win logic. I test observed GSC impression volume as a public-safe exposure measure; I am not recreating the private flag or treating its decision as a label. Hypothesis: higher-volume buckets have higher later decline rates. Fixed buckets: 100–299, 300–999, 1,000–2,999, and 3,000+ impressions. Volume can also justify a review's potential relevance even if it does not predict decline reliably.

**Signal B — prior week-to-week impression change:** Hypothesis: an already-falling page is more likely to show a later decline than a stable or rising page. Fixed buckets: at most −20%, between −20% and +20%, and above +20%. This is **March 8–14 versus March 1–7**, not the target-window trend. It is therefore not the label-derived `trend_pct` trap from the starter data.

Each table prints page count **n**, positives, contributing clients, page-weighted decline rate, and an equal-client rate. The second rate helps reveal client-composition effects; it does not remove every confounder.

In [1]:
# Install only missing libraries; no credentials appear in this command.
import importlib.util
import subprocess
import sys

packages = {"duckdb": "duckdb",
            "sklearn": "scikit-learn", "pandas": "pandas"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import os
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

MONTH = "2026-03"
REPO_ID = "FlyRank/internship-warehouse"
REVISION = "50cbf7c3909d07be4d1b5906b4d09e882e5acbf2"
FILENAME = f"fact_content_daily_performance/month={MONTH}/data_0.parquet"

# Find a repo root if present; otherwise use the current Colab folder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "docs/ml-intern-dataset-and-lane-guide.md").is_file()), Path.cwd())
local_candidates = [
    root / "data/warehouse" / f"month={MONTH}" / "data_0.parquet",
    root / "data/warehouse" / FILENAME,
]
month_file = next((p for p in local_candidates if p.is_file()), None)

if month_file is None:
    if importlib.util.find_spec("huggingface_hub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import get_token, hf_hub_download
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    token = token or get_token()
    if not token:
        raise RuntimeError("Warehouse access is required: set HF_TOKEN securely, "
                           "use a local Hugging Face login, or supply the March parquet file. "
                           "Do not paste a token into this notebook.")
    month_file = Path(hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset", filename=FILENAME,
        revision=REVISION, token=token, cache_dir=str(root / "data/warehouse_cache"),
    ))
    del token

con = duckdb.connect()
con.execute("SET threads = 2")
con.execute("SET memory_limit = '1GB'")
temp_folder = root / "data/warehouse_cache/duckdb_tmp"
temp_folder.mkdir(parents=True, exist_ok=True)
con.execute("SET temp_directory = ?", [str(temp_folder)])
# This is a local cached file, so repeat verification does not rescan Hugging Face.
con.read_parquet(str(month_file)).create_view("march_raw")
required_columns = {
    "report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
    "ga4_data_available", "gsc_impressions", "gsc_clicks", "gsc_avg_position",
}
observed_columns = set(con.table("march_raw").columns)
assert required_columns <= observed_columns, "Warehouse schema differs; inspect before proceeding."
print("Loaded the cached March partition; required field names verified.")

Loaded the cached March partition; required field names verified.


In [2]:
from sklearn.model_selection import GroupShuffleSplit

# This feature query reads only past dates: no future counts or coverage in eligibility.
past_query = """
SELECT client_hash_id, content_hash_id,
       COUNT(*) AS feature_days,
       SUM(gsc_impressions) AS feature_impressions,
       SUM(gsc_clicks) AS feature_clicks,
       SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-07') AS first_week_impressions,
       SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-08') AS second_week_impressions
FROM march_raw
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL AND gsc_clicks IS NOT NULL
  AND gsc_impressions >= 0 AND gsc_clicks >= 0
GROUP BY client_hash_id, content_hash_id
HAVING feature_days = 14 AND feature_impressions >= 100
ORDER BY client_hash_id, content_hash_id
"""
past = con.execute(past_query).df()
past["prior_week_change_pct"] = (
    100 * (past.second_week_impressions - past.first_week_impressions)
    / past.first_week_impressions.replace(0, np.nan)
)
past["decision_date"] = "2026-03-18"
assert not past.duplicated(["client_hash_id", "content_hash_id", "decision_date"]).any()
assert not past.empty

# Future measurements are held separately and used solely for labels/evaluation.
outcome_query = """
SELECT client_hash_id, content_hash_id, COUNT(*) AS outcome_days,
       SUM(gsc_impressions) AS outcome_impressions
FROM march_raw
WHERE report_date BETWEEN DATE '2026-03-18' AND DATE '2026-03-31'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL AND gsc_clicks IS NOT NULL
  AND gsc_impressions >= 0 AND gsc_clicks >= 0
GROUP BY client_hash_id, content_hash_id
HAVING outcome_days = 14
"""
outcomes = con.execute(outcome_query).df()
labeled = past.merge(outcomes, on=["client_hash_id", "content_hash_id"],
                     how="inner", validate="one_to_one")
labeled = labeled.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
labeled["future_decline_proxy"] = (
    labeled.outcome_impressions.lt(0.8 * labeled.feature_impressions).astype("int64")
)
train_idx, test_idx = next(GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
).split(labeled, groups=labeled.client_hash_id))
development = labeled.iloc[train_idx].copy()
test_clients = set(labeled.iloc[test_idx].client_hash_id)
assert set(development.client_hash_id).isdisjoint(test_clients)

print(f"Past-only scoring population: {len(past):,} pages across {past.client_hash_id.nunique()} clients.")
print(f"Complete-case labeled population: {len(labeled):,} pages; "
      f"development: {len(train_idx):,}, test: {len(test_idx):,}.")
print("Unknown future coverage excludes rows from evaluation, not from the action queue.")

Past-only scoring population: 60,534 pages across 33 clients.
Complete-case labeled population: 57,459 pages; development: 14,070, test: 43,389.
Unknown future coverage excludes rows from evaluation, not from the action queue.


In [3]:
def bucket_table(frame, bucket_column):
    table = frame.groupby(bucket_column, observed=False).agg(
        n=("future_decline_proxy", "size"),
        positives=("future_decline_proxy", "sum"),
        clients=("client_hash_id", "nunique"),
        decline_rate=("future_decline_proxy", "mean"),
    )
    client_rates = frame.groupby(
        ["client_hash_id", bucket_column], observed=True
    ).future_decline_proxy.mean().groupby(level=1, observed=False).mean()
    table["equal_client_decline_rate"] = client_rates
    return table.reset_index()

development["volume_bucket"] = pd.cut(
    development.feature_impressions, bins=[99, 299, 999, 2999, np.inf],
    labels=["100-299", "300-999", "1000-2999", "3000+"],
)
volume_audit = bucket_table(development, "volume_bucket")
assert volume_audit.n.sum() == len(development)
print("Signal A: pre-decision volume; n is the number of pages in each bucket.")
volume_audit.round(4)

Signal A: pre-decision volume; n is the number of pages in each bucket.


,volume_bucket,n,positives,clients,decline_rate,equal_client_decline_rate
0,100-299,4825,1446,24,0.2997,0.3162
1,300-999,5342,1713,20,0.3207,0.4291
2,1000-2999,2583,866,18,0.3353,0.4323
3,3000+,1320,432,15,0.3273,0.3483


**Signal A verdict: MIXED.** Page-weighted later decline rates are roughly 30.0%, 32.1%, 33.5%, and 32.7%; the highest-volume bucket is not the highest-risk bucket. Equal-client rates also rise and then fall, so volume is not a reliable monotonic risk signal here. I retain a volume floor and modest log-volume weighting as **exposure/impact policy**, not as evidence that high volume causes decline. That distinction saves the rule from claiming a risk relationship the table does not support.

In [4]:
development["prior_change_bucket"] = pd.cut(
    development.prior_week_change_pct, bins=[-np.inf, -20, 20, np.inf],
    labels=["down <= -20%", "between -20% and +20%", "up > +20%"],
)
trend_audit = bucket_table(development, "prior_change_bucket")
assert trend_audit.n.sum() == len(development)
print("Signal B: pre-decision week-to-week change; n is the number of pages in each bucket.")
trend_audit.round(4)

Signal B: pre-decision week-to-week change; n is the number of pages in each bucket.


,prior_change_bucket,n,positives,clients,decline_rate,equal_client_decline_rate
0,down <= -20%,5080,1937,22,0.3813,0.4040
1,between -20% and +20%,4959,1577,22,0.3180,0.3010
2,up > +20%,4031,943,22,0.2339,0.2972


**Signal B verdict: CONFIRMED.** In this development slice, later decline rates are about 38.1% for already-falling pages, 31.8% for stable pages, and 23.4% for rising pages. The down bucket also has the highest equal-client rate. This supports prior decline as an observational review signal here; it does not establish causation, persistence across months, or the best cutoff.

**One rule, in plain words:** Review a page if it has at least **300 impressions in March 1–14** and a **week-to-week fall of at least 20%** within that input window. Rank it by `log(1 + input impressions) × fractional prior-week drop`, so exposure matters gently while a larger drop increases priority. Give every positive-score recommendation the one reason code `visible_prior_week_decline` and action `review_for_possible_refresh`.

The 300-impression floor and −20% cutoff are provisional review policies, not universal truths, and were not optimized on test outcomes. A score is unitless priority, not a probability or expected recoverable traffic. Zero score means the rule makes no recommendation; those rows do not enter the exported action queue.

## 2. Build the ranked queue and freeze the baseline

The score function accepts only two measured pre-decision inputs. It never sees labels, outcome counts, outcome coverage, product flags, or IDs. IDs break ties and provide grouping context outside the score function. The CSV contains positive-score candidates only, with both global and within-client ranks.

In [5]:
MIN_RULE_IMPRESSIONS = 300
MAX_PRIOR_CHANGE_PCT = -20.0
RULE_INPUTS = ["feature_impressions", "prior_week_change_pct"]
REASON_CODE = "visible_prior_week_decline"
ACTION = "review_for_possible_refresh"

def score_rule(inputs):
    assert list(inputs.columns) == RULE_INPUTS
    visible = inputs.feature_impressions.ge(MIN_RULE_IMPRESSIONS)
    falling = inputs.prior_week_change_pct.le(MAX_PRIOR_CHANGE_PCT)
    score = pd.Series(0.0, index=inputs.index, name="score")
    eligible = visible & falling
    score.loc[eligible] = (
        np.log1p(inputs.loc[eligible, "feature_impressions"])
        * (-inputs.loc[eligible, "prior_week_change_pct"] / 100)
    )
    return score

scored_population = past.copy()
scored_population["score"] = score_rule(scored_population[RULE_INPUTS])
queue = scored_population.loc[scored_population.score.gt(0)].copy()
queue = queue.sort_values(["score", "client_hash_id", "content_hash_id"],
                          ascending=[False, True, True]).reset_index(drop=True)
queue["global_rank"] = np.arange(1, len(queue) + 1)
queue["client_rank"] = queue.groupby("client_hash_id").cumcount() + 1
queue["reason_code"] = REASON_CODE
queue["action"] = ACTION
queue["confidence_note"] = "directional review candidate; refresh benefit unproven"
queue_columns = [
    "global_rank", "client_rank", "client_hash_id", "content_hash_id", "decision_date",
    "feature_impressions", "first_week_impressions", "second_week_impressions",
    "prior_week_change_pct", "score", "reason_code", "action", "confidence_note",
]
queue = queue[queue_columns]
output_dir = root / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "baseline_action_score.csv"
queue.to_csv(csv_path, index=False)
assert len(queue) >= 10
assert queue.reason_code.nunique() == 1 and queue.action.nunique() == 1
assert not {"future_decline_proxy", "outcome_impressions", "outcome_days"}.intersection(queue.columns)
assert queue.score.is_monotonic_decreasing
print(f"Wrote {len(queue):,} ranked review candidates across {queue.client_hash_id.nunique()} clients.")
print("CSV: work/outputs/baseline_action_score.csv (regenerated locally; not committed).")
queue.head(10)[["global_rank", "content_hash_id", "feature_impressions",
                "prior_week_change_pct", "score", "reason_code", "action"]]

Wrote 14,853 ranked review candidates across 28 clients.
CSV: work/outputs/baseline_action_score.csv (regenerated locally; not committed).


,global_rank,content_hash_id,feature_impressions,prior_week_change_pct,score,reason_code,action
0,1,content_9c057b66c30a3abb,83770.0,-99.965369,11.331916,visible_prior_week_decline,review_for_possible_refresh
1,2,content_34a70fea29d15f24,67555.0,-93.768084,10.427679,visible_prior_week_decline,review_for_possible_refresh
2,3,content_945d6ff91386c817,44156.0,-95.843751,10.250975,visible_prior_week_decline,review_for_possible_refresh
3,4,content_0c5606abaaab3178,24427.0,-99.679658,10.071120,visible_prior_week_decline,review_for_possible_refresh
4,5,content_1642f339bd6e7c8d,35924.0,-94.452932,9.907346,visible_prior_week_decline,review_for_possible_refresh
5,6,content_1ff6231687184dec,11932.0,-99.120730,9.304525,visible_prior_week_decline,review_for_possible_refresh
6,7,content_bb2a9972810ddd72,22345.0,-90.776224,9.090697,visible_prior_week_decline,review_for_possible_refresh
7,8,content_0bca6d9a85a9b408,15510.0,-94.165814,9.086346,visible_prior_week_decline,review_for_possible_refresh
8,9,content_84c64d54522c71b0,12231.0,-95.782209,9.014840,visible_prior_week_decline,review_for_possible_refresh
9,10,content_2d3ea336a4467aa5,8962.0,-97.717416,8.893126,visible_prior_week_decline,review_for_possible_refresh


**Evaluation policy for Week 5:** Mean per-client Precision@20 uses the same complete-case test population, target, seed-42 client split, and eligible test clients as Week 3. Rank all eligible labeled pages by the rule score, including zero-score ties resolved by content ID, so the population is comparable to a learned model's full ranking. The CSV action threshold is separate: zero-score pages are not recommended. I print positive-score coverage of the top-20 slots so this distinction stays visible.

I show both the test page-level base rate and the matched equal-client random expectation. Precision@20 excludes clients with fewer than 20 labeled pages. This is one historical client holdout, not a final future-month benchmark. The Week 5 comparison must keep the recorded baseline and evaluate on these same rows; earlier Week 3 quick-tree results are not a final model comparison.

In [6]:
import hashlib
import json

test = labeled.iloc[test_idx].copy()
test["score"] = score_rule(test[RULE_INPUTS])
test_sizes = test.groupby("client_hash_id").content_hash_id.transform("size")
metric_population = test.loc[test_sizes.ge(20)].copy()
metric_ranked = metric_population.sort_values(
    ["client_hash_id", "score", "content_hash_id"], ascending=[True, False, True]
)
metric_top20 = metric_ranked.groupby("client_hash_id").head(20)
assert metric_top20.groupby("client_hash_id").size().eq(20).all()
precision20 = metric_top20.groupby("client_hash_id").future_decline_proxy.mean().mean()
random_expected_precision20 = metric_population.groupby(
    "client_hash_id"
).future_decline_proxy.mean().mean()
positive_score_slots = int(metric_top20.score.gt(0).sum())

split_receipt = (
    test[["client_hash_id", "content_hash_id", "future_decline_proxy"]]
    .sort_values(["client_hash_id", "content_hash_id"])
    .to_csv(index=False)
)
metrics = {
    "baseline_version": "w04_v1", "lane": "Lane 2: Refresh / Content Opportunity Scoring",
    "warehouse_revision": REVISION, "month": MONTH, "decision_date": "2026-03-18",
    "feature_window": ["2026-03-01", "2026-03-14"],
    "outcome_window": ["2026-03-18", "2026-03-31"],
    "target": "outcome impressions < 0.8 * feature impressions; 14 valid days per window",
    "input_names": RULE_INPUTS, "rule_min_impressions": MIN_RULE_IMPRESSIONS,
    "rule_max_prior_change_pct": MAX_PRIOR_CHANGE_PCT,
    "score_formula": "log1p(feature_impressions) * (-prior_week_change_pct / 100) when both gates pass; else 0",
    "reason_code": REASON_CODE, "action": ACTION,
    "past_only_scoring_pages": len(past), "labeled_pages": len(labeled),
    "queue_pages": len(queue), "queue_clients": int(queue.client_hash_id.nunique()),
    "development_pages": len(train_idx), "test_pages": len(test_idx),
    "split_seed": 42, "holdout_client_fraction": 0.25,
    "train_clients": sorted(development.client_hash_id.unique().tolist()),
    "test_clients": sorted(test.client_hash_id.unique().tolist()),
    "test_population_sha256": hashlib.sha256(split_receipt.encode("utf-8")).hexdigest(),
    "test_page_base_rate": float(test.future_decline_proxy.mean()),
    "metric_k": 20, "metric_clients": int(metric_top20.client_hash_id.nunique()),
    "small_test_clients_excluded": int(test.client_hash_id.nunique() - metric_top20.client_hash_id.nunique()),
    "mean_client_precision_at_20": float(precision20),
    "expected_random_mean_client_precision_at_20": float(random_expected_precision20),
    "positive_score_top20_slots": positive_score_slots, "total_top20_slots": len(metric_top20),
    "volume_verdict": "MIXED", "prior_change_verdict": "CONFIRMED",
    "volume_audit": volume_audit.to_dict(orient="records"),
    "prior_change_audit": trend_audit.to_dict(orient="records"),
    "limitation": "One March cutoff and complete-case retrospective labels; no causal refresh-benefit estimate.",
}
metrics_path = output_dir / "w04_baseline_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(f"Test page-level base rate: {test.future_decline_proxy.mean():.2%}.")
print(f"Expected random mean client Precision@20: {random_expected_precision20:.2%}.")
print(f"Baseline mean client Precision@20: {precision20:.2%}, across "
      f"{metric_top20.client_hash_id.nunique()} clients.")
print(f"Positive-score recommendations occupy {positive_score_slots} of "
      f"{len(metric_top20)} evaluation slots; zero-score ties are not action recommendations.")
print("Wrote aggregate receipt: work/outputs/w04_baseline_metrics.json.")
pd.DataFrame({
    "reference": ["Test page base rate", "Matched random expectation", "Frozen rule Precision@20"],
    "value": [test.future_decline_proxy.mean(), random_expected_precision20, precision20],
}).round(4)

Test page-level base rate: 38.25%.
Expected random mean client Precision@20: 39.81%.
Baseline mean client Precision@20: 61.43%, across 7 clients.
Positive-score recommendations occupy 119 of 140 evaluation slots; zero-score ties are not action recommendations.
Wrote aggregate receipt: work/outputs/w04_baseline_metrics.json.


,reference,value
0,Test page base rate,0.3825
1,Matched random expectation,0.3981
2,Frozen rule Precision@20,0.6143


## 3. Top-10 skeptical review

Every recommendation below has action **review_for_possible_refresh**, reason **visible_prior_week_decline**, and **directional confidence only**. The reason is known at the decision moment; no later labels were used to explain these picks.

1. **content_9c057b66c30a3abb:** Review for a possible refresh; 83,770 input-window impressions and a −99.97% prior-week change make it rank first, but only 29 impressions remain in week 2—an earlier exposure spike or tracking/indexing problem could make a content refresh the wrong response.
2. **content_34a70fea29d15f24:** Review for a possible refresh; 67,555 impressions and −93.77% change give high exposure plus a large drop—if its client's many simultaneous drops share a technical cause, an individual rewrite could waste time.
3. **content_945d6ff91386c817:** Review for a possible refresh; 44,156 impressions and −95.84% change warrant investigation—if demand for the topic fell, page freshness may not explain the loss.
4. **content_0c5606abaaab3178:** Review for a possible refresh; 24,427 impressions and −99.68% change score highly despite just 78 week-2 impressions—if demand moved to a redirected or consolidated sibling page, refreshing this item could be inappropriate.
5. **content_1642f339bd6e7c8d:** Review for a possible refresh; 35,924 impressions and −94.45% change combine exposure with severe movement—if a temporary indexing or site-level issue caused it, the correct action may be technical investigation.
6. **content_1ff6231687184dec:** Review for a possible refresh; 11,932 impressions and −99.12% change put it high in the queue, with only 104 week-2 impressions—if week 1 was an unusual short-lived demand spike, the score could overstate a lasting problem.
7. **content_bb2a9972810ddd72:** Review for a possible refresh; 22,345 impressions and −90.78% change justify a closer look—if broader search-result or demand changes caused the fall, editing this page may not help.
8. **content_0bca6d9a85a9b408:** Review for a possible refresh; 15,510 impressions and −94.17% change trigger the rule—if the shared-client pattern is driven by migration or a site-wide issue, this should be investigated as a group.
9. **content_84c64d54522c71b0:** Review for a possible refresh; 12,231 impressions and −95.78% change suggest a large loss—if another related item absorbed its exposure, the recommendation could create unnecessary competing content.
10. **content_2d3ea336a4467aa5:** Review for a possible refresh; 8,962 impressions and −97.72% change earn a high score, with 200 week-2 impressions—if the content was deliberately withdrawn or redirected, reviving it may conflict with the site's intended action.

These are hypotheses to check, not diagnoses. The displayed table verifies that the written review still matches the actual regenerated top ten.

In [7]:
top10 = queue.head(10).copy()
REVIEWED_CONTENT_IDS = [
    "content_9c057b66c30a3abb", "content_34a70fea29d15f24", "content_945d6ff91386c817",
    "content_0c5606abaaab3178", "content_1642f339bd6e7c8d", "content_1ff6231687184dec",
    "content_bb2a9972810ddd72", "content_0bca6d9a85a9b408", "content_84c64d54522c71b0",
    "content_2d3ea336a4467aa5",
]
assert top10.content_hash_id.tolist() == REVIEWED_CONTENT_IDS, (
    "Top ten changed: update the written review before submitting."
)
print("Verified ten reviewed IDs against the regenerated queue.")
print(f"Largest-client share of the global top ten: "
      f"{top10.client_hash_id.value_counts().max()} of 10.")
top10[["global_rank", "client_hash_id", "content_hash_id", "feature_impressions",
       "second_week_impressions", "prior_week_change_pct", "score"]].round(4)

Verified ten reviewed IDs against the regenerated queue.
Largest-client share of the global top ten: 9 of 10.


,global_rank,client_hash_id,content_hash_id,feature_impressions,second_week_impressions,prior_week_change_pct,score
0,1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83770.0,29.0,-99.9654,11.3319
1,2,client_62f4a7e64f5e0096,content_34a70fea29d15f24,67555.0,3963.0,-93.7681,10.4277
2,3,client_62f4a7e64f5e0096,content_945d6ff91386c817,44156.0,1762.0,-95.8438,10.2510
3,4,client_62f4a7e64f5e0096,content_0c5606abaaab3178,24427.0,78.0,-99.6797,10.0711
4,5,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,35924.0,1888.0,-94.4529,9.9073
5,6,client_62f4a7e64f5e0096,content_1ff6231687184dec,11932.0,104.0,-99.1207,9.3045
6,7,client_62f4a7e64f5e0096,content_bb2a9972810ddd72,22345.0,1887.0,-90.7762,9.0907
7,8,client_62f4a7e64f5e0096,content_0bca6d9a85a9b408,15510.0,855.0,-94.1658,9.0863
8,9,client_62f4a7e64f5e0096,content_84c64d54522c71b0,12231.0,495.0,-95.7822,9.0148
9,10,client_62f4a7e64f5e0096,content_2d3ea336a4467aa5,8962.0,200.0,-97.7174,8.8931


## 4. Weak picks, limits, and leakage checks

**Weak picks found:** Ranks 1 and 4 lean heavily on week-1 exposure even though their week-2 counts are just 29 and 78. They may deserve technical/measurement investigation, but the evidence for a useful content refresh is weak. Rank 6 has a similar risk with 104 week-2 impressions. The rule does not distinguish historical exposure from demand still worth pursuing; I document that flaw instead of changing the rule after seeing test results.

**Concentration risk:** Nine of the global top ten belong to the same pseudonymized client. That could reflect a shared site-wide event and can consume review capacity with repeated versions of one issue. The CSV includes within-client ranks, and the evaluation uses client-level Precision@20. Neither establishes the cause; a later playbook should consider grouped investigation and capacity limits.

**Limits:** Volume's verdict is MIXED, and its weighting is an impact policy, not a validated decline-probability estimate. March alone cannot establish seasonality, future-month stability, or persistence. Complete later coverage is required only for labels, creating an evaluation selection bias for sparse-history pages. The assumed reporting buffer is not proof of historical ingestion time. Zero-score ties affect the full-ranking metric and are explicitly separated from actionable queue rows. No comparison here proves that refreshing content causes recovery.

**Baseline freeze:** Keep this score, thresholds, target, split, and tie-breaking policy unchanged for Week 5. A learned model must be evaluated on the same recorded test population and per-client metric; a higher number on a different slice is not a win. June remains the sealed final-month data, and is never loaded in this notebook.

In [8]:
# A practical leakage check: score only the two allowlisted pre-decision columns.
assert RULE_INPUTS == ["feature_impressions", "prior_week_change_pct"]
assert "outcome" not in past_query.lower() and "2026-03-18" not in past_query
for forbidden in ["future_decline_proxy", "outcome_impressions", "outcome_days",
                  "trend_pct", "trend_direction", "health_score", "priority_score", "leak_target_copy"]:
    assert forbidden not in RULE_INPUTS and forbidden not in queue.columns

# Altering labels cannot alter scores: the function receives no outcome columns.
audit_copy = labeled.copy()
audit_copy["future_decline_proxy"] = 1 - audit_copy.future_decline_proxy
audit_copy["outcome_impressions"] = 0
np.testing.assert_allclose(
    score_rule(labeled[RULE_INPUTS]), score_rule(audit_copy[RULE_INPUTS])
)
written_queue = pd.read_csv(csv_path)
assert len(written_queue) == len(queue)
assert written_queue.content_hash_id.tolist() == queue.content_hash_id.tolist()
assert written_queue.reason_code.nunique() == 1 and written_queue.action.nunique() == 1
saved_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
assert saved_metrics["queue_pages"] == len(queue)
assert saved_metrics["mean_client_precision_at_20"] == float(precision20)
print("Verified: past-only queue eligibility; two allowlisted inputs; label changes leave scores unchanged.")
print("Verified: CSV regenerates, one reason code/action, metrics receipt matches, top ten reviewed.")

Verified: past-only queue eligibility; two allowlisted inputs; label changes leave scores unchanged.
Verified: CSV regenerates, one reason code/action, metrics receipt matches, top ten reviewed.


## 5. Self-check

- [x] Confirmed Lane 2 for the remaining capstone work.
- [x] Two development-client bucket tables show n and measured outcomes, with one verdict each.
- [x] The volume check explicitly links to the flag-related exposure signal on the assignment card.
- [x] One transparent rule has a score, exactly one reason code, and an action label.
- [x] The action queue uses only pre-decision measurements and does not filter by future coverage.
- [x] The notebook wrote `work/outputs/baseline_action_score.csv` and verified its contents.
- [x] Ten real top-ranked items have an action, measured reason, and a possible failure mode.
- [x] Weak picks, shared-client concentration, metric coverage, and slice limitations are explained.
- [x] Baseline Precision@20 and base rates are reported; the aggregate JSON receipt is saved.
- [x] All cells ran in order with outputs retained and no future or label-derived score inputs.

**Submission:** Commit this executed notebook and `work/outputs/w04_baseline_metrics.json`. Keep the generated CSV and parquet data out of Git. Push, check the notebook outputs on GitHub, then submit the repository URL on this assignment card.